In [1]:

import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1' # Ép chạy 1 luồng duy nhất
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("../").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.utils.utils import load_csv
from src.features.sbert_features import SBERTFeatureExtractor
from src.features.metadata_features import MetadataFeatureExtractor
from src.features.feature_union import FeatureUnion
from src.models.train_lightgbm import LightGBMTrainer

from src.config import (
    CLEANED_TRAIN_PATH,
    CLEANED_TEST_PATH,
    TRAIN_LABEL_PATH,
    FINAL_TRAIN_SBERT_PATH,
    FINAL_TEST_SBERT_PATH,
    SUBMISSION_GBM_PATH,
    SUBMISSION_KNN_PATH
)

/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = load_csv(CLEANED_TRAIN_PATH)
test_df = load_csv(CLEANED_TEST_PATH)
labels_df = load_csv(TRAIN_LABEL_PATH)
y_train = load_csv(TRAIN_LABEL_PATH)
y_train = y_train - 1
y_train = np.asarray(y_train).ravel()

# FEATURE EXTRACTION (SBERT + METADATA)
sbert_ext = SBERTFeatureExtractor(model_name='all-MiniLM-L6-v2')
meta_ext = MetadataFeatureExtractor()
union = FeatureUnion(is_transformers=True)

print("🚀 Extracting Train Features...")
X_train, _ = union.fit_transform(sbert_ext, meta_ext, train_df)

print("🚀 Extracting Test Features...")
X_test = union.transform(sbert_ext, meta_ext, test_df)

# Save features for later use
np.save(FINAL_TRAIN_SBERT_PATH, X_train)
np.save(FINAL_TEST_SBERT_PATH, X_test)
print(f"✔ Saved features to {FINAL_TRAIN_SBERT_PATH.parent}")

/Users/nhatnam/Documents/DM_252/Assignment/venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded SBERT model: all-MiniLM-L6-v2 on cpu
🚀 Extracting Train Features...


Batches: 100%|██████████| 78/78 [00:35<00:00,  2.21it/s]


🚀 Extracting Test Features...


Batches: 100%|██████████| 19/19 [00:08<00:00,  2.19it/s]

✔ Saved features to /Users/nhatnam/Documents/DM_252/Assignment/data/processed


In [3]:
print(f"\n====================")
print(f"MODEL: LIGHTGBM + SBERT")
print(f"====================")

# Khởi tạo Trainer
trainer = LightGBMTrainer(params={
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,          # 1. Giảm sâu để học chậm, tránh nhảy vọt vào các lỗ hổng
    'num_leaves': 5,                # 2. Rất quan trọng: Cực thấp để cây không thể chia quá nhiều
    'max_depth': 3,                 # 3. Khóa chặt độ sâu để tránh cây quá phức tạp
    'min_data_in_leaf': 30,         # 4. Ép mỗi lá phải có ít nhất 30 mẫu mới được chia (tránh học vẹt)
    'max_bin': 15,                  # 5. Giảm bin xuống cực thấp (giúp generalize tốt hơn và chống crash)
    
    'feature_fraction': 0.4,        # 6. Mỗi cây chỉ được chọn 40% số cột để học (tạo sự đa dạng)
    'bagging_fraction': 0.6,        # 7. Lấy 60% dữ liệu ngẫu nhiên để train từng cây
    'bagging_freq': 5,
    
    'lambda_l1': 0.5,               # 8. Regularization L1 (loại bỏ bớt feature nhiễu)
    'lambda_l2': 1.0,               # 9. Regularization L2 (giữ trọng số nhỏ lại)
    
    'random_state': 42,
    'num_threads': 1,
    'force_col_wise': True,
    'verbose': -1
})

# Huấn luyện 5-fold CV

macro_f1_cv = trainer.train(X_train, y_train, n_splits=5)

print(f"CV Macro F1-Score: {macro_f1_cv:.4f}")


MODEL: LIGHTGBM + SBERT
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid's multi_logloss: 1.38624
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[496]	valid's multi_logloss: 1.40471
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[479]	valid's multi_logloss: 1.39352
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[437]	valid's multi_logloss: 1.38953
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[484]	valid's multi_logloss: 1.367

Overall Macro F1: 0.2671
CV Macro F1-Score: 0.2671


In [4]:
# 1. Dự đoán từ mô hình (đang trả về nhãn 0-4)
y_test_pred = trainer.predict(X_test)

# 2. CỘNG LẠI 1 để đưa về nhãn gốc (1-5)
y_test_pred_final = y_test_pred + 1

# 3. Tạo DataFrame submission
# Đảm bảo test_df đã được load lại nếu bạn lỡ xóa nó trước đó
submission = pd.DataFrame({
    "id": test_df["id"],
    "Label": y_test_pred_final
})

# 4. Lưu file
submission.to_csv(SUBMISSION_GBM_PATH, index=False)
print(f"✔ Saved GBM submission → {SUBMISSION_GBM_PATH}")

# 5. Kiểm tra phân bổ nhãn (Lúc này phải từ 1 đến 5)
print("\nPredicted labels distribution (Should be 1-5):")
print(submission['Label'].value_counts().sort_index())

✔ Saved GBM submission → /Users/nhatnam/Documents/DM_252/Assignment/data/submission/submission_sbert_gbm.csv

Predicted labels distribution (Should be 1-5):
Label
1    482
2     26
3     24
4     33
5     31
Name: count, dtype: int64


In [5]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import f1_score, classification_report

print(f"\n====================")
print(f"MODEL: K-NEAREST NEIGHBORS (KNN)")
print(f"====================")

# 1. Xử lý nhãn (Đưa về 0-4 để train)
y_train_knn = y_train

# 2. Khởi tạo mô hình KNN
# n_neighbors=5: Lấy 5 hàng xóm gần nhất
# metric='cosine': CỰC KỲ QUAN TRỌNG cho vector SBERT (đo góc thay vì đo độ dài)
# weights='distance': Hàng xóm nào càng gần thì 'tiếng nói' càng có trọng lượng
knn_model = KNeighborsClassifier(
    n_neighbors=5, 
    weights='distance', 
    metric='cosine'
)

# 3. Đánh giá nhanh bằng 5-Fold CV (Để xem điểm có khá hơn LightGBM không)
cv_scores = cross_val_score(knn_model, X_train, y_train_knn, cv=5, scoring='f1_macro')
print(f"✔ KNN (k=5) CV Macro F1-Score: {cv_scores.mean():.4f}")

# 4. Huấn luyện mô hình cuối cùng trên toàn bộ dữ liệu Train
print("✔ Training final KNN model...")
knn_model.fit(X_train, y_train_knn)

# 5. DỰ ĐOÁN VÀ TẠO FILE SUBMISSION
print("✔ Predicting on Test set...")
y_test_pred_knn = knn_model.predict(X_test)

# QUAN TRỌNG: Cộng lại 1 để đưa nhãn về khoảng 1-5 theo yêu cầu hệ thống
y_test_pred_final = y_test_pred_knn + 1

# Load lại ID (nếu chưa có) và lưu file

submission_knn = pd.DataFrame({
    "id": test_df["id"],
    "Label": y_test_pred_final
})

# Lưu file submission

submission_knn.to_csv(SUBMISSION_KNN_PATH, index=False)

print(f"✔ Saved KNN submission → {SUBMISSION_KNN_PATH}")
print("\nPredicted labels distribution (Should be 1-5):")
print(submission_knn['Label'].value_counts().sort_index())


MODEL: K-NEAREST NEIGHBORS (KNN)
✔ KNN (k=5) CV Macro F1-Score: 0.1365
✔ Training final KNN model...
✔ Predicting on Test set...
✔ Saved KNN submission → /Users/nhatnam/Documents/DM_252/Assignment/data/submission/submission_sbert_knn.csv

Predicted labels distribution (Should be 1-5):
Label
1    288
2     98
3    105
4     61
5     44
Name: count, dtype: int64
